# Sequence Models: RNN, LSTM, GRU, Seq2Seq and Attention

Notebook for **Chapter 10** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 Sequences and the Recurrent Neural Network

In [ ]:
import numpy as np
# A tiny RNN: 1 input number, 2 hidden numbers.  h_t = tanh(Wx*x_t + Wh@h_{t-1} + b)
Wx = np.array([[0.5], [-0.3]])             # shape (2,1)
Wh = np.array([[0.1, 0.4], [0.2, -0.1]])   # shape (2,2)
b  = np.array([0.0, 0.1])
xs = [1.0, 2.0, -1.0]                      # a sequence of 3 inputs
h = np.zeros(2)
for t, x in enumerate(xs, 1):
    a = Wx[:, 0] * x + Wh @ h + b          # pre-activation
    h = np.tanh(a)
    print(f"t={t} x={x:5.1f}  a={a.round(4)}  h={h.round(4)}")

t=1 x=  1.0  a=[ 0.5 -0.2]  h=[ 0.4621 -0.1974]
t=2 x=  2.0  a=[ 0.9673 -0.3878]  h=[ 0.7475 -0.3695]
t=3 x= -1.0  a=[-0.573   0.5864]  h=[-0.5176  0.5273]


In [ ]:
series = np.array([10, 12, 13, 15, 14, 16, 18, 17, 19, 21.])   # e.g. daily orders in a small shop
T = 4                                                          # look at 4 days, predict day 5
X = np.array([series[i:i+T] for i in range(len(series) - T)])  # shape (N, T)
y = series[T:]
print("X shape", X.shape, "y shape", y.shape)
print(X[0], "->", y[0]); print(X[1], "->", y[1]); print(X[-1], "->", y[-1])
print("model input shape (batch, time, features):", X[:, :, None].shape)

X shape (6, 4) y shape (6,)
[10. 12. 13. 15.] -> 14.0
[12. 13. 15. 14.] -> 16.0
[16. 18. 17. 19.] -> 21.0
model input shape (batch, time, features): (6, 4, 1)


In [ ]:
# How much does the first input still influence the last hidden state?  Measure |d h_T / d h_0|
rng = np.random.default_rng(0)
def grad_norm(scale, T, n=16):
    Q, _ = np.linalg.qr(rng.normal(size=(n, n)))
    Wh = Q * scale                          # every singular value equals `scale`
    h = rng.normal(size=n) * 0.5; J = np.eye(n)
    for t in range(T):
        a = Wh @ h; h = np.tanh(a)
        J = np.diag(1 - h**2) @ Wh @ J      # chain rule: dh_t/dh_0 = diag(1-h^2) Wh dh_{t-1}/dh_0
    return np.linalg.norm(J, 2)
print("steps   scale 0.9      scale 1.0      scale 1.5")
for T in (1, 5, 10, 20, 50, 100):
    print(f"{T:5d}   " + "   ".join(f"{grad_norm(s, T):12.3e}" for s in (0.9, 1.0, 1.5)))

steps   scale 0.9      scale 1.0      scale 1.5
    1      8.992e-01      9.990e-01      1.500e+00
    5      4.809e-01      7.872e-01      3.310e+00
   10      2.991e-01      6.636e-01      5.795e+00
   20      9.257e-02      4.845e-01      1.173e+01
   50      4.308e-03      2.762e-01      1.619e+02
  100      2.183e-05      2.630e-01      2.964e+03


In [ ]:
# Train a small RNN (hidden 6) to say whether the sum of 8 numbers is positive.  Backprop through time + gradient check.
rng = np.random.default_rng(0)
H, T = 6, 8
P = {"Wx": rng.normal(0, 0.5, (H, 1)), "Wh": rng.normal(0, 0.5, (H, H)) / np.sqrt(H), "b": np.zeros(H),
     "Wo": rng.normal(0, 0.5, (1, H)), "bo": np.zeros(1)}
sig = lambda a: 1 / (1 + np.exp(-a))
def forward(P, x):                                   # x: (T,)
    hs = [np.zeros(H)]
    for t in range(T): hs.append(np.tanh(P["Wx"][:, 0] * x[t] + P["Wh"] @ hs[-1] + P["b"]))
    return hs, sig(P["Wo"] @ hs[-1] + P["bo"])[0]
def loss_grad(P, x, y):
    hs, p = forward(P, x)
    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    g = {k: np.zeros_like(v) for k, v in P.items()}
    d = p - y                                        # dL/d(output score) for sigmoid + cross-entropy
    g["Wo"] += d * hs[-1][None, :]; g["bo"] += d
    dh = d * P["Wo"][0]
    for t in range(T, 0, -1):                        # go backwards through time
        da = dh * (1 - hs[t] ** 2)
        g["Wx"][:, 0] += da * x[t - 1]; g["Wh"] += np.outer(da, hs[t - 1]); g["b"] += da
        dh = P["Wh"].T @ da                          # pass the gradient to the previous step
    return loss, g
# gradient check on one example
x0 = rng.normal(size=T); y0 = float(x0.sum() > 0)
_, g = loss_grad(P, x0, y0); worst = 0
for k in P:
    for idx in np.ndindex(*P[k].shape):
        old = P[k][idx]; P[k][idx] = old + 1e-6; lp = loss_grad(P, x0, y0)[0]
        P[k][idx] = old - 1e-6; lm = loss_grad(P, x0, y0)[0]; P[k][idx] = old
        num = (lp - lm) / 2e-6; worst = max(worst, abs(num - g[k][idx]) / (abs(num) + abs(g[k][idx]) + 1e-12))
print("gradient check, worst relative error:", f"{worst:.2e}")
X = rng.normal(size=(600, T)); Y = (X.sum(1) > 0).astype(float)
Xte = rng.normal(size=(300, T)); Yte = (Xte.sum(1) > 0).astype(float)
acc = lambda X, Y: np.mean([(forward(P, x)[1] > 0.5) == y for x, y in zip(X, Y)])
print("before training: test accuracy", round(float(acc(Xte, Yte)), 3))
lr = 0.02
def clip(g, m=1.0):                              # gradient clipping by global norm
    n = np.sqrt(sum((v ** 2).sum() for v in g.values()))
    return g if n <= m else {k: v * m / n for k, v in g.items()}
for epoch in range(1, 31):
    tot = 0
    for i in rng.permutation(len(X)):
        l, g = loss_grad(P, X[i], Y[i]); g = clip(g); tot += l
        for k in P: P[k] -= lr * g[k]
    if epoch in (1, 5, 10, 20, 30): print(f"epoch {epoch:2d}  mean loss {tot/len(X):.4f}  test accuracy {acc(Xte, Yte):.3f}")

gradient check, worst relative error: 2.86e-08
before training: test accuracy 0.737
epoch  1  mean loss 0.4434  test accuracy 0.870
epoch  5  mean loss 0.3024  test accuracy 0.907
epoch 10  mean loss 0.2548  test accuracy 0.933
epoch 20  mean loss 0.2187  test accuracy 0.940
epoch 30  mean loss 0.2075  test accuracy 0.937


## 10.2 LSTM, GRU, Bidirectional and Stacked RNNs

In [ ]:
sig = lambda a: 1 / (1 + np.exp(-a))
# One LSTM unit with one input number.  Each gate: weight on x, weight on h_prev, bias.
W = {"f": (0.6, 0.4, 0.5), "i": (0.3, 0.2, 0.0), "o": (0.5, 0.1, 0.0), "g": (0.7, 0.1, -0.2)}
x, h_prev, c_prev = 1.0, 0.5, 0.8
pre = {k: wx * x + wh * h_prev + b for k, (wx, wh, b) in W.items()}
f, i, o = sig(pre["f"]), sig(pre["i"]), sig(pre["o"]); g = np.tanh(pre["g"])
c = f * c_prev + i * g          # forget part of the old memory, add part of the new candidate
h = o * np.tanh(c)              # the output is a filtered view of the memory
for k, v in pre.items(): print(f"pre-activation {k}: {v:.3f}")
print(f"forget f={f:.4f}  input i={i:.4f}  output o={o:.4f}  candidate g={g:.4f}")
print(f"new cell c = {f:.4f}*{c_prev} + {i:.4f}*{g:.4f} = {c:.4f}")
print(f"new hidden h = {o:.4f} * tanh({c:.4f}) = {h:.4f}")

pre-activation f: 1.300
pre-activation i: 0.400
pre-activation o: 0.550
pre-activation g: 0.550
forget f=0.7858  input i=0.5987  output o=0.6341  candidate g=0.5005
new cell c = 0.7858*0.8 + 0.5987*0.5005 = 0.9283
new hidden h = 0.6341 * tanh(0.9283) = 0.4628


In [ ]:
# LSTM memory path: dc_T/dc_0 is the product of the forget gates along the way
for f in (0.5, 0.9, 0.95, 0.99):
    print(f"forget gate {f}: after 10 steps {f**10:.4f}   after 50 steps {f**50:.3e}   after 100 steps {f**100:.3e}")
# Parameter counts: n_gates * (hidden*(input+hidden) + hidden)
d, h = 10, 20
for name, gates in (("plain RNN", 1), ("GRU", 3), ("LSTM", 4)):
    print(f"{name:9s} input {d}, hidden {h}: {gates * (h * (d + h) + h)} parameters")
print("bidirectional LSTM:", 2 * 4 * (h * (d + h) + h), " 2-layer LSTM (layer 2 input is h):", 4 * (h * (d + h) + h) + 4 * (h * (h + h) + h))

forget gate 0.5: after 10 steps 0.0010   after 50 steps 8.882e-16   after 100 steps 7.889e-31
forget gate 0.9: after 10 steps 0.3487   after 50 steps 5.154e-03   after 100 steps 2.656e-05
forget gate 0.95: after 10 steps 0.5987   after 50 steps 7.694e-02   after 100 steps 5.921e-03
forget gate 0.99: after 10 steps 0.9044   after 50 steps 6.050e-01   after 100 steps 3.660e-01
plain RNN input 10, hidden 20: 620 parameters
GRU       input 10, hidden 20: 1860 parameters
LSTM      input 10, hidden 20: 2480 parameters
bidirectional LSTM: 4960  2-layer LSTM (layer 2 input is h): 5760


In [ ]:
sig = lambda a: 1 / (1 + np.exp(-a))
# One GRU unit, same style: (weight on x, weight on h_prev, bias)
W = {"z": (0.4, 0.3, 0.0), "r": (0.5, 0.2, 0.1), "n": (0.8, 0.6, 0.0)}
x, h_prev = 1.0, 0.5
z = sig(W["z"][0] * x + W["z"][1] * h_prev + W["z"][2])          # update gate
r = sig(W["r"][0] * x + W["r"][1] * h_prev + W["r"][2])          # reset gate
n = np.tanh(W["n"][0] * x + W["n"][1] * (r * h_prev) + W["n"][2])  # candidate uses the reset memory
h = (1 - z) * n + z * h_prev                                       # blend new candidate with old memory
print(f"update z={z:.4f}  reset r={r:.4f}  candidate n={n:.4f}")
print(f"new hidden h = (1-{z:.4f})*{n:.4f} + {z:.4f}*{h_prev} = {h:.4f}")

update z=0.6341  reset r=0.6682  candidate n=0.7618
new hidden h = (1-0.6341)*0.7618 + 0.6341*0.5 = 0.5958


## 10.3 Seq2Seq: Encoder-Decoder, Teacher Forcing and Beam Search

In [ ]:
# Next-token probabilities P(next | previous) for a toy decoder. "<s>" starts, every path ends after 2 tokens.
P = {"<s>": {"A": 0.6, "B": 0.4},
     "A": {"C": 0.34, "D": 0.33, "E": 0.33},
     "B": {"F": 0.9, "G": 0.1}}
def greedy():
    seq, p, tok = [], 1.0, "<s>"
    for _ in range(2):
        tok, q = max(P[tok].items(), key=lambda kv: kv[1]); seq.append(tok); p *= q
    return seq, p
def beam(width):
    beams = [([], 1.0, "<s>")]
    for step in range(2):
        cand = [(s + [t], p * q, t) for s, p, last in beams for t, q in P[last].items()]
        beams = sorted(cand, key=lambda b: -b[1])[:width]
        print(f"  width {width}, step {step+1} keeps:", [("".join(s), round(p, 3)) for s, p, _ in beams])
    return beams[0][0], beams[0][1]
print("greedy:", greedy())
print("beam 1:", beam(1)); print("beam 2:", beam(2))

greedy: (['A', 'C'], 0.20400000000000001)
  width 1, step 1 keeps: [('A', 0.6)]
  width 1, step 2 keeps: [('AC', 0.204)]
beam 1: (['A', 'C'], 0.20400000000000001)
  width 2, step 1 keeps: [('A', 0.6), ('B', 0.4)]
  width 2, step 2 keeps: [('BF', 0.36), ('AC', 0.204)]
beam 2: (['B', 'F'], 0.36000000000000004)


## 10.4 Attention, and Why Transformers Won

In [ ]:
enc = np.array([[1., 0.], [0., 1.], [1., 1.]])        # encoder states for 3 source words (2 numbers each)
q = np.array([2., 0.])                                # decoder state asking "what should I look at now?"
scores = enc @ q
def softmax(v): e = np.exp(v - v.max()); return e / e.sum()
w = softmax(scores); context = w @ enc
print("scores", scores, "-> weights", w.round(4), "sum", w.sum().round(4))
print("context vector", context.round(4))
ws = softmax(scores / np.sqrt(2))                     # scaled by sqrt(d), d = 2
print("scaled weights", ws.round(4), "context", (ws @ enc).round(4))

scores [2. 0. 2.] -> weights [0.4683 0.0634 0.4683] sum 1.0
context vector [0.9366 0.5317]
scaled weights [0.4458 0.1084 0.4458] context [0.8916 0.5542]


In [ ]:
# PyTorch equivalents (not run here)
import torch.nn as nn
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True, bidirectional=False)
# input (batch, time, 10) -> output (batch, time, 20), (h_n, c_n)
gru = nn.GRU(10, 20, batch_first=True)
# clip gradients before the optimizer step:
# torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

## 10.5 Beam Search by Hand, and State-Space Models

Section 10.3 described beam search in a paragraph. Here you run it on numbers, so you can see exactly when it beats greedy decoding. Then we look at the other idea that revisits the RNN: state-space models, which are recurrences trained like convolutions.

In [ ]:
import math
# P(next token | previous token); tokens: a, b, c, </s>
P = {
    "<s>": {"a": 0.5, "b": 0.4, "c": 0.1, "</s>": 0.0},
    "a":   {"a": 0.3, "b": 0.3, "c": 0.0, "</s>": 0.4},
    "b":   {"a": 0.05, "b": 0.0, "c": 0.9, "</s>": 0.05},
    "c":   {"a": 0.0, "b": 0.0, "c": 0.0, "</s>": 1.0},
}
# greedy
seq, p = ["<s>"], 1.0
while seq[-1] != "</s>":
    tok = max(P[seq[-1]], key=P[seq[-1]].get)
    p *= P[seq[-1]][tok]; seq.append(tok)
print("greedy", seq[1:], "prob", round(p, 3))

# beam search, width 2
beams = [(["<s>"], 0.0)]            # (tokens, log-prob)
done = []
for step in range(4):
    cand = []
    for toks, lp in beams:
        for t, q in P[toks[-1]].items():
            if q > 0: cand.append((toks + [t], lp + math.log(q)))
    cand.sort(key=lambda x: -x[1])
    beams = []
    for toks, lp in cand[:2]:
        (done if toks[-1] == "</s>" else beams).append((toks, lp))
    print("step", step + 1, [(" ".join(t[1:]), round(math.exp(lp), 3)) for t, lp in cand[:2]])
    if not beams: break
best = max(done, key=lambda x: x[1])
print("beam best", best[0][1:], "prob", round(math.exp(best[1]), 3))

greedy ['a', '</s>'] prob 0.2
step 1 [('a', 0.5), ('b', 0.4)]
step 2 [('b c', 0.36), ('a </s>', 0.2)]
step 3 [('b c </s>', 0.36)]
beam best ['b', 'c', '</s>'] prob 0.36


In [ ]:
import numpy as np
x = np.array([1.0, 0.0, 0.0, 2.0, 0.0])
A, B, C = 0.5, 1.0, 2.0                      # one state, fixed ("linear time-invariant") SSM
h, ys = 0.0, []
for xt in x:
    h = A * h + B * xt                      # h_t = A h_{t-1} + B x_t
    ys.append(C * h)
print("recurrent y   ", np.round(ys, 4))
k = np.array([C * A**j * B for j in range(len(x))])   # impulse response of the recurrence
y_conv = np.convolve(x, k)[:len(x)]
print("kernel        ", np.round(k, 4))
print("convolution y ", np.round(y_conv, 4))

# selective version: the decay depends on the input (a gate), as in Mamba
gate = lambda xt: 0.9 if xt != 0 else 0.2   # toy rule: keep memory when something arrives
h, ys = 0.0, []
for xt in x:
    h = gate(xt) * h + B * xt
    ys.append(C * h)
print("selective y   ", np.round(ys, 4))

recurrent y    [2.    1.    0.5   4.25  2.125]
kernel         [2.    1.    0.5   0.25  0.125]
convolution y  [2.    1.    0.5   4.25  2.125]
selective y    [2.     0.4    0.08   4.072  0.8144]


## Chapter recap

## Practice set

**1.** (easy) One RNN unit, `Wx = 1`, `Wh = 0.5`, `b = 0`, `h₀ = 0`, inputs `1, 1`. Find `h₂`. **Answer:** `h₁ = tanh(1) = 0.7616`; `h₂ = tanh(1 + 0.5 × 0.7616) = tanh(1.3808) = 0.8811`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch10.html